# Exploration des résultats — Modèle vs In-situ

Lit les sorties déjà produites par `eval_dtod_quantile.py` et `compare_other_models_vs_insitu.py` (pas de recalcul SWORD/métriques ici) pour :

1. Afficher les 10 meilleures stations (par NSE) pour un modèle/fréquence donnés
2. Tracer le graphe détaillé (in-situ / modèle / altimétrie) pour une station + une année choisies
3. Un aperçu global de la distribution des métriques

⚠️ Fixé sur `SOURCE = "hwnext"` — adapté au schéma BDD unifié (`measurements`/`orthometric_height`).

In [ ]:
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parents[1]))  # ajuste si le notebook n'est pas 2 niveaux sous la racine
from Pipeline_data.Step1_Niveau_deau.config_step1 import INSITU_DB_PATH

%matplotlib inline

## Paramètres — à ajuster ici

In [ ]:
SOURCE = "hwnext"          # fixé, on ne gère pas dahiti dans ce notebook
MODEL_LABEL = "DtoD96"     # ex: "DtoD80", "DtoD90", "DtoD96", "Quantile80", ...
FREQ = "27j"               # "10j" ou "27j"

# DtoD* -> sous-dossier "DtoD", Quantile* -> sous-dossier "Quantille" (comme les scripts sources)
MODEL_SUBDIR = "Quantille" if "Quantile" in MODEL_LABEL else "DtoD"

RESIDUS_DIR = Path(f"./Models_Testing/{MODEL_SUBDIR}/residus")
METRICS_CSV = RESIDUS_DIR / f"metrics_{MODEL_LABEL}_{SOURCE}_{FREQ}_sword_insitu.csv"
RESIDUALS_CSV = RESIDUS_DIR / f"residuals_{MODEL_LABEL}_{SOURCE}_{FREQ}.csv"

INSITU_DB = INSITU_DB_PATH
DATE_MIN, DATE_MAX = "2016-01-01", "2025-12-31"
WINDOW_DAYS = {"10j": 5, "27j": 14}[FREQ]

print(f"Métriques : {METRICS_CSV}  (existe: {METRICS_CSV.exists()})")
print(f"Résidus   : {RESIDUALS_CSV}  (existe: {RESIDUALS_CSV.exists()})")
print(f"BDD in-situ : {INSITU_DB}")

## Fonctions utilitaires

In [ ]:
_cache_ins = {}


def get_insitu_series(code_sta: str) -> pd.DataFrame:
    """
    Série in-situ brute — schéma unifié (measurements/orthometric_height),
    au lieu de l'ancien mesures_insitu/h_med_wsh.
    """
    if code_sta not in _cache_ins:
        conn = sqlite3.connect(INSITU_DB)
        df = pd.read_sql(
            """
            SELECT measure_date AS date, orthometric_height AS wl
            FROM measurements
            WHERE station_code = ? AND is_valid = 1
              AND measure_date >= ? AND measure_date <= ?
            ORDER BY measure_date
            """,
            conn, params=(code_sta, DATE_MIN, DATE_MAX),
        )
        conn.close()
        df["date"] = pd.to_datetime(df["date"])
        _cache_ins[code_sta] = df.dropna(subset=["wl"])
    return _cache_ins[code_sta]


def zscore_params(values):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if len(v) < 2:
        return 0.0, 1.0
    mu, sigma = v.mean(), v.std()
    return mu, (sigma if sigma > 0 else 1.0)


def align_insitu_to_dates(dates, df_ins, window_days):
    out = np.full(len(dates), np.nan)
    if df_ins.empty:
        return out
    id_ = np.array(df_ins["date"].values, dtype="datetime64[D]")
    iv = df_ins["wl"].values
    for i, d in enumerate(np.array(dates, dtype="datetime64[D]")):
        diff = np.abs((id_ - d).astype(float))
        idx = int(np.argmin(diff))
        if diff[idx] <= window_days:
            out[i] = iv[idx]
    return out

## 1. Top 10 stations (par NSE)

In [ ]:
df_metrics = pd.read_csv(METRICS_CSV)
df_metrics["station"] = df_metrics["station"].astype(str)

top10 = (
    df_metrics.dropna(subset=["NSE"])
    .sort_values("NSE", ascending=False)
    .head(10)
    [["station", "insitu_code", "dist_insitu_km", "connectivity_validated",
      "n_pairs", "NSE", "KGE", "RMSE", "R2"]]
    .reset_index(drop=True)
)

print(f"Top 10 stations — {MODEL_LABEL} [{FREQ}]")
top10

## 2. Plot détaillé — station + année au choix

Change `STATION` et `YEAR` ci-dessous puis relance la cellule.

In [ ]:
STATION = top10.iloc[0]["station"]   # <-- remplace par le code station de ton choix
YEAR = 2022                          # <-- remplace par l'année de ton choix

print(f"Station : {STATION}  |  Année : {YEAR}")

In [ ]:
def plot_station_year(station: str, year: int, freq: str = FREQ):
    df_res = pd.read_csv(RESIDUALS_CSV)
    df_res["station"] = df_res["station"].astype(str)
    df_res["date"] = pd.to_datetime(df_res["date"])

    sub_daily = df_res[df_res["station"] == station].sort_values("date")
    if sub_daily.empty:
        print(f"⚠ Aucune donnée pour la station {station}")
        return

    row_met = df_metrics[df_metrics["station"] == station]
    if row_met.empty:
        print(f"⚠ Station {station} absente de {METRICS_CSV.name} (pas de match insitu)")
        insitu_code, global_nse = None, np.nan
    else:
        insitu_code = row_met["insitu_code"].iloc[0]
        global_nse = row_met["NSE"].iloc[0]

    # z-score partagé obs/pred, calculé sur les dates avec observation réelle
    aligned = sub_daily.dropna(subset=["obs", "pred"])
    obs_mu, obs_sigma = zscore_params(aligned["obs"])
    pred_mu, pred_sigma = zscore_params(aligned["pred"])
    sub_daily = sub_daily.assign(
        obs_z=(sub_daily["obs"] - obs_mu) / obs_sigma,
        pred_z=(sub_daily["pred"] - pred_mu) / pred_sigma,
    )

    df_ins = get_insitu_series(insitu_code).copy() if insitu_code else pd.DataFrame()
    if len(df_ins):
        aligned_ins = align_insitu_to_dates(aligned["date"].values, df_ins, WINDOW_DAYS)
        ins_mu, ins_sigma = zscore_params(aligned_ins)
        df_ins["wl_z"] = (df_ins["wl"] - ins_mu) / ins_sigma

    sub_year = sub_daily[sub_daily["date"].dt.year == year]
    ins_year = df_ins[df_ins["date"].dt.year == year] if len(df_ins) else pd.DataFrame()
    if sub_year.empty:
        print(f"⚠ Aucune donnée pour {station} en {year}")
        return

    obs_year = sub_year.dropna(subset=["obs"])
    pred_year = sub_year.dropna(subset=["pred"])
    pairs_year = sub_year.dropna(subset=["obs", "pred"])
    year_nse = (
        1 - np.sum((pairs_year["obs"] - pairs_year["pred"]) ** 2)
        / np.sum((pairs_year["obs"] - pairs_year["obs"].mean()) ** 2)
        if len(pairs_year) >= 3 and np.sum((pairs_year["obs"] - pairs_year["obs"].mean()) ** 2) > 0
        else np.nan
    )

    fig, ax = plt.subplots(figsize=(11, 4.5))

    if len(ins_year):
        ax.plot(ins_year["date"], ins_year["wl_z"], "-", color="#229954",
                linewidth=1.1, alpha=0.55, label="Insitu (quotidien, z-score)", zorder=1)

    if len(pred_year):
        ax.plot(pred_year["date"], pred_year["pred_z"], "-", color="#C0392B",
                linewidth=1.3, alpha=0.85, label=f"Modèle {MODEL_LABEL} (quotidien, z-score)", zorder=2)

    ax.plot(obs_year["date"], obs_year["obs_z"], "o", color="#1B4F72", markersize=6.5,
            markeredgecolor="white", markeredgewidth=0.7,
            label="Altimétrie (obs, z-score)", zorder=3)

    nse_txt = f"NSE {year} (aux dates obs) = {year_nse:.2f}" if pd.notna(year_nse) else "NSE (n insuffisant)"
    ax.set_title(f"{station}  ·  {freq}  ·  {year}", fontsize=12, fontweight="bold", loc="left")
    ax.text(0.99, 1.03, nse_txt, transform=ax.transAxes, ha="right", va="bottom",
            fontsize=9.5, style="italic")
    ax.text(0.01, -0.16,
            f"NSE global (toute la période) = {global_nse:.3f}  ·  insitu = {insitu_code}  ·  "
            f"séries centrées-réduites (z-score, indépendamment par courbe)",
            transform=ax.transAxes, ha="left", va="top", fontsize=7.5, color="#7F8C8D")

    ax.set_ylabel("Niveau d'eau — z-score")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(0, -0.06),
              ncol=3, handletextpad=0.5)

    fig.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


plot_station_year(STATION, YEAR)

## 3. Aperçu global des métriques (toutes stations)

In [ ]:
metrics_to_plot = ["NSE", "KGE", "RMSE", "R2"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(16, 4))
fig.suptitle(f"Distribution des métriques — {MODEL_LABEL} [{FREQ}]  (n={len(df_metrics)} stations)",
             fontsize=13, fontweight="bold")

for ax, m in zip(axes, metrics_to_plot):
    vals = df_metrics[m].dropna()
    ax.boxplot(vals, vert=True, patch_artist=True,
               boxprops=dict(facecolor="#AED6F1"), medianprops=dict(color="#1B4F72", linewidth=2))
    ax.set_title(m, fontsize=11, fontweight="bold")
    ax.set_xticks([])
    ax.grid(True, alpha=0.3, axis="y", linestyle="--")
    ax.text(0.5, -0.08, f"médiane={vals.median():.3f}", transform=ax.transAxes,
            ha="center", fontsize=9, color="#7F8C8D")

plt.tight_layout()
plt.show()

# Complétude de la connectivité SWORD (info secondaire mais utile à voir d'un coup d'oeil)
pct_valid = 100 * df_metrics["connectivity_validated"].mean()
print(f"Stations avec sélection in-situ validée par connectivité SWORD : {pct_valid:.0f}%")